# Cleaning data

In [ ]:
import pandas as pd

df = pd.read_csv("/content/LLM_Data_2 - Grad_Project_Data.csv")
df.head()

,Question,Answer
0,Did ancient Egyptian women have a high social ...,"Yes, they enjoyed a relatively high social sta..."
1,How did property inheritance work in ancient E...,All landed property was passed down through th...
2,Why was property passed down through the femal...,It was based on the assumption that maternity ...
3,Did ancient Egyptian women have to wear veils ...,"No, unlike the women of ancient Greece, they e..."
4,How were male and female guests seated at form...,Married guests sat together in pairs on fine c...


In [ ]:
def format_row(row):
    q = str(row["Question"]).strip()
    a = str(row["Answer"]).strip()

    return f"Question: {q} Answer: {a}"

df["text"] = df.apply(format_row, axis=1)

In [ ]:
df = df[df["text"].str.len() > 20]
df = df.drop_duplicates(subset=["text"])

In [ ]:
df[["text"]].to_csv("cleaned_data.csv", index=False)

In [ ]:
df["text"].iloc[0]

'Question: Did ancient Egyptian women have a high social status? Answer: Yes, they enjoyed a relatively high social status and could exert influence outside their domestic roles.'

# Qwen 2.5

In [ ]:
!pip install transformers sentence-transformers faiss-cpu accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd

df = pd.read_csv("cleaned_data.csv")
texts = df["text"].tolist()

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(texts, show_progress_bar=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/396 [00:00<?, ?it/s]

In [ ]:
import faiss
import numpy as np

index = faiss.IndexFlatL2(len(embeddings[0]))
index.add(np.array(embeddings))

In [ ]:
def retrieve(query, k=2):
    q_emb = embedder.encode([query])
    distances, indices = index.search(q_emb, k * 2)

    results = [texts[i] for i in indices[0]]

    filtered = [r for r in results if any(word.lower() in r.lower() for word in query.split())]

    return filtered[:k] if filtered else results[:k]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def answer_question(query):
    context_list = retrieve(query)
    context = "\n".join(context_list)

    prompt = f"""
    You are a friendly and knowledgeable Egyptian tourist guide.

    Answer the question using ONLY the information provided in the context.

    Style:
    - Speak naturally and clearly like a guide.
    - Use 2-3 sentences.

    Strict Rules:
    - Do NOT add explanations or interpretations.
    - Do NOT add any information not directly written in the context.
    - Do NOT justify or comment on the information.
    - Do NOT repeat the question.

    Context:
    {context}

    Visitor Question: {query}
    Guide Answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.2
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    answer = response.split("Guide Answer:")[-1]
    answer = answer.split("Visitor Question:")[0].strip()

    if not answer.endswith((".", "!", "?")):
        answer = answer.rsplit(".", 1)[0] + "."

    return answer

In [ ]:
answer_question("Who was Hatshepsut?")

'Hatshepsut was both the stepmother and aunt of Thutmose III, serving as his regent during part of his early rule. According to Roger Dunn’s article, she reigned from around 1504–1483 BCE. The Cambridge Ancient History described her in the mid-20th century with negative views, portraying her negatively as "a conniving," "shrewd," "ambitious," and an "unscrupulous" woman.'

In [ ]:
answer_question("When did Hatshepsut reign?")

'Hatshepsut reigned from around 1504 to 1483 B.C.E., spanning over twenty-one years during Egypt’s New Kingdom era. To clarify, her rule lasted approximately 21 years.'

In [ ]:
answer_question("What monuments did Hatshepsut build?")

'Yes, she built impressive monuments such as the Great Hypostyle Hall at Karnak Temple and the magnificent temple complex at Deir el-Bahri. These include the famous Mortuary Temple with its distinctive valley temple and mortuary chapel. To visit these sites today gives you an idea of how grand her reign was through monumental architecture.'

In [ ]:
answer_question("Who was Ramesses II?")

'Ramesses II, also known as Usermaatre-setepenre, reigned for over six decades and is one of Egypt’s most famous pharaohs. He built numerous temples and monuments across his kingdom to commemorate his victories and divine lineage. His mother was Queen Nefertari, making him part of an illustrious royal family.'

In [ ]:
answer_question("Why is Ramesses II called “the Great”?")

'Ramesses II was given this title because he achieved numerous military victories and ruled for an exceptionally long time—over 60 years. His monuments were grander than those of his predecessors, showcasing his power and legacy. The epithet "Great" reflects both his achievements and enduring impact on Egypt\'s history.'

In [ ]:
answer_question("Who was Ramesses II and what were his achievements?")

'Ramesses II was an iconic pharaoh known for his monumental architecture across Egypt. He also distinguished himself through significant victories in battle, making him one of the greatest kings in ancient history. His reign marked great advancements both politically and militarily. To visit some of his impressive structures, consider exploring places like Abu Simbel and Karnak Temple. These sites bear witness to his legacy with their grandeur and scale.'

In [ ]:
answer_question("What does the Sphinx represent?")

'The Sphinx is believed to represent the pharaoh Khafre. To further elaborate, its lion body likely symbolizes strength and might associated with divine kingship.'

In [ ]:
answer_question("Can you describe the Sphinx?")

"The Great Sphinx stands as an awe-inspiring monument, its colossal size dwarfing everything around it. With the body of a mighty lion and the head adorned by the distinctive Nemes headdress, this ancient guardian watches over Giza's pyramids majestically."